# Two fixed frequency-band features: LFC/HFC, participant LOSO

This notebook contains all extraction, evaluation, reporting and save/reload code.
It needs no project Python modules, but **does need the existing private data files**.
It reuses the exact filtered 500-sample windows, reviewed temporal labels and subject
grouping from the earlier models. It does not filter raw recordings again.

Two model inputs only: **LFC power (20–46.7 Hz)** and **HFC power (150–350 Hz)**.
These are absolute band powers obtained by integrating a Welch PSD; they are not
peak spectral amplitudes or an exact replication of an unspecified amplitude
estimator. Band RMS (square root of power) is saved for descriptive interpretation
but is **not** an additional model input. No ratio, log transform, relative-power
variant, significance selection or feature-count search is tested.

Fixed model: training-median imputation → training StandardScaler → RBF SVC(C=1,
gamma='scale'). Fifteen participant LOSO fits plus one final fit; no inner search,
class weights/resampling, calibration or threshold tuning. No GPU is required.
This is an exploratory follow-up after inspecting previous results. Stage labels
are proxies, not independent fatigue truth. Different results cannot be attributed
solely to physiological band specificity. The literature's exact bands were used
in diaphragm EMG; applicability to dynamic rectus femoris is being explored.

References for provenance, not a claim of exact replication:
- https://pubmed.ncbi.nlm.nih.gov/457515/
- Gross D, Grassino A, Ross WR, Macklem PT (1979), Electromyogram pattern of
  diaphragmatic fatigue: https://doi.org/10.1152/jappl.1979.46.1.1
- De Luca CJ (1993), Use of the surface EMG signal for performance evaluation of
  back muscles: https://doi.org/10.1002/mus.880160216
- https://pmc.ncbi.nlm.nih.gov/articles/PMC8914984/
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html

Copy only this notebook to the existing private Drive modelling folder. The
`cnn_data/` waveform bundle used by notebook06 and original `data/` feature bundle
must already exist there. Keep data, outputs and executed notebooks private.
Real model training is to be run in authorized private Colab, not the laptop.

## 1. Private paths and environment

Set paths before installing anything. If a new runtime lacks the pinned numerical
stack, set INSTALL_REQUIREMENTS=True once, run, restart that same kernel, restore
False and run from the first cell. The installation cell deliberately stops after
installation. No project requirement file or Python module is needed by this notebook.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from importlib.metadata import version

# Set this only when the notebook is opened outside the cloned project.
PROJECT_ROOT_OVERRIDE = os.environ.get('EMG_PROJECT_ROOT')
start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in [start, *start.parents]
    if (p / 'project_paths.py').is_file() and (p / 'modelling/core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the cloned project, or set EMG_PROJECT_ROOT to its root.')
sys.path.insert(0, str(PROJECT_ROOT))
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
    raise RuntimeError('Restart this kernel, restore INSTALL_REQUIREMENTS=False, then run from the first cell.')
for package, expected in {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0'}.items():
    actual=version(package)
    print(package,actual)
    if actual!=expected:raise RuntimeError('Install requirements in this environment and restart the kernel.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
probe=subprocess.run([sys.executable,'-c','import numpy, pandas, scipy.sparse, sklearn; print("Fresh-process imports OK")'],capture_output=True,text=True)
print(probe.stdout)
if probe.returncode:raise RuntimeError(probe.stderr)
print('Project:',PROJECT_ROOT)
print('Python:',sys.version.split()[0])

import hashlib, platform, time, warnings, shutil
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError
SIGNAL_DIR = PROJECT_ROOT / 'modelling/cnn_data'
FEATURE_DIR = PROJECT_ROOT / 'modelling/data'
OUTPUT_ROOT = PROJECT_ROOT / 'modelling/outputs'
PINS = {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0',
        'matplotlib':'3.10.6','joblib':'1.5.3','threadpoolctl':'3.6.0'}
for package, expected in PINS.items():
    if version(package) != expected:
        raise RuntimeError('Install pinned requirements and restart this kernel.')


In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import signal
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    cohen_kappa_score, roc_auc_score, precision_recall_curve, average_precision_score,
    auc, confusion_matrix, roc_curve)
import joblib
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
from IPython.display import display, Image

CONFIG=dict(seed=20261018,fs_hz=1000.,window_samples=500,welch_nperseg=256,
    welch_noverlap=128,welch_nfft=256,lfc_hz=[20.,46.7],hfc_hz=[150.,350.],
    integration='trapezoid with linearly interpolated PSD at exact band edges',
    C=1.,gamma='scale',cache_mb=128,tol=.001,max_iter=-1,final_refit=True,
    expected_subjects=15,expected_sets=139,expected_windows=9215)
FEATURES=['LFC_power_20_46p7','HFC_power_150_350']
METRICS=['accuracy','precision_macro','recall_macro','f1_macro','f1_class_0','f1_class_1',
    'cohen_kappa','roc_auc','pr_auc','average_precision']
MODEL_KEY='lfc_hfc_fixed_rbf'
display(pd.Series(CONFIG,name='Fixed choices').to_frame())


## 2. Frozen numerical methods and synthetic checks

Welch uses a periodic Hann window, 256 samples, 128 internal overlap, nfft256,
constant detrending, one-sided density scaling and mean averaging. At1000 Hz its
bin spacing is3.90625 Hz. Exact band-edge interpolation does not improve spectral
resolution or make46.7 Hz an independently resolved bin. The original20–450 Hz
offline filter remains in force; its lower cutoff affects the lower band edge.

Power integrates the linear PSD (not dB). No MVC, resting reference, whole-set
normalization or fitted population statistic enters extraction. Computing these
fixed per-window descriptors before splitting does not fit a learned transform.
Tests below use invented signals only and do not train a model.

In [ ]:
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,default=str)+'\n',encoding='utf-8')

def integrate_band(f,psd,lo,hi):
    f=np.asarray(f,dtype=float);psd=np.asarray(psd,dtype=float)
    if f.ndim!=1 or psd.shape!=f.shape or not np.all(np.diff(f)>0):
        raise ValueError('Expected matching one-dimensional increasing frequency/PSD arrays.')
    if not np.isfinite(psd).all() or (psd<0).any() or not f[0]<=lo<hi<=f[-1]:
        raise ValueError('Invalid PSD or band bounds.')
    inside=(f>lo)&(f<hi)
    grid=np.r_[lo,f[inside],hi]
    values=np.r_[np.interp(lo,f,psd),psd[inside],np.interp(hi,f,psd)]
    return float(np.trapezoid(values,grid))

def window_powers(x,fs=1000.):
    x=np.asarray(x,dtype=float)
    if x.shape!=(500,) or not np.isfinite(x).all():
        raise ValueError('Expected one finite, original 500-sample window.')
    hann=signal.windows.hann(256,sym=False)
    f,p=signal.welch(x,fs=fs,window=hann,nperseg=256,noverlap=128,nfft=256,
        detrend='constant',return_onesided=True,scaling='density',average='mean')
    return np.array([integrate_band(f,p,20.,46.7),integrate_band(f,p,150.,350.)])

def make_pipeline():
    return Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
        ('scale',StandardScaler()),('model',SVC(kernel='rbf',C=1.,gamma='scale',
            probability=False,cache_size=128,tol=.001,max_iter=-1,random_state=20261018))])

def evaluate(y,pred,score):
    p,r,f,s=precision_recall_fscore_support(y,pred,labels=[0,1],zero_division=0)
    result=dict(accuracy=accuracy_score(y,pred),precision_macro=p.mean(),recall_macro=r.mean(),
        f1_macro=f.mean(),f1_class_0=f[0],f1_class_1=f[1],cohen_kappa=cohen_kappa_score(y,pred,labels=[0,1]),
        roc_auc=np.nan,pr_auc=np.nan,average_precision=np.nan,n_windows=len(y),support_0=int(s[0]),support_1=int(s[1]))
    if len(np.unique(y))==2:
        precision,recall,_=precision_recall_curve(y,score,pos_label=1)
        result.update(roc_auc=roc_auc_score(y,score),pr_auc=auc(recall,precision),
            average_precision=average_precision_score(y,score))
    pc=pd.DataFrame(dict(class_label=[0,1],class_name=['early','late'],precision=p,recall=r,f1=f,support=s))
    return result,pc

# Synthetic analytical integration, gain and band-specificity checks; no fitting.
f_test=np.arange(501,dtype=float)
assert np.isclose(integrate_band(f_test,np.full(501,2.),20.,46.7),53.4)
t_test=np.arange(500)/1000.
low_test=window_powers(np.sin(2*np.pi*31.25*t_test))
high_test=window_powers(np.sin(2*np.pi*250*t_test))
assert low_test[0]>100*low_test[1] and high_test[1]>100*high_test[0]
assert np.isclose(low_test[0],.5,rtol=.03) and np.isclose(high_test[1],.5,rtol=.03)
np.testing.assert_allclose(window_powers(3*np.sin(2*np.pi*31.25*t_test)),9*low_test,rtol=1e-9,atol=1e-15)
np.testing.assert_array_equal(window_powers(np.zeros(500)),[0.,0.])
assert evaluate([0,0,1,1],[0,0,1,1],[0,0,1,1])[0]['f1_macro']==1.
print('Synthetic spectral/metric checks passed. No model was fitted.')


## 3. Load and verify the existing private waveform bundle

Checks cover the waveform/metadata checksums, original feature table checksum,
window order/IDs, participant/set/leg/label identity, sampling rate and the original
15/139/9215 dimensions. Missing files stop execution; there is no automatic data
upload, waveform reconstruction, exclusion change or label revision.

In [ ]:
paths={'signal_manifest':SIGNAL_DIR/'signal_manifest.json',
    'signals':SIGNAL_DIR/'signal_windows.npz','metadata':SIGNAL_DIR/'window_metadata.csv',
    'model_manifest':FEATURE_DIR/'data_manifest.json','model_windows':FEATURE_DIR/'model_windows.csv'}
for name,path in paths.items():
    if not path.is_file():raise FileNotFoundError(f'Missing {name}: {path}. Reuse the private notebook06 waveform bundle.')
source=json.loads(paths['signal_manifest'].read_text())
model_source=json.loads(paths['model_manifest'].read_text())
assert sha(paths['signals'])==source['signals_sha256']
assert sha(paths['metadata'])==source['metadata_sha256']
assert sha(paths['model_windows'])==source['source_model_data_sha256']==model_source['data_sha256']
windows=pd.read_csv(paths['metadata'])
reference=pd.read_csv(paths['model_windows'],usecols=['window_id','subject_id','recording_id','leg','Label'])
pd.testing.assert_frame_equal(windows[reference.columns],reference,check_dtype=False)
with np.load(paths['signals'],allow_pickle=False) as z:
    signals=z['signals'];ids=z['window_id']
assert signals.shape==(9215,500) and np.isfinite(signals).all()
assert np.array_equal(ids,windows.window_id.to_numpy()) and windows.window_id.is_unique
assert windows.subject_id.nunique()==15 and windows.recording_id.nunique()==139
assert windows.Label.isin([0,1]).all() and windows.groupby('subject_id').Label.nunique().eq(2).all()
assert np.isclose(windows.fs_used_hz.to_numpy(),1000.,rtol=1e-8,atol=1e-6).all()
assert 'S009_REC01' not in set(windows.recording_id)
display(windows.groupby('Label').size().rename('windows').to_frame())
display(windows.groupby(['subject_id','Label']).size().unstack(fill_value=0))
print('Verified original reviewed windows; both legs and all sets remain grouped by participant.')


## 4. Extract the two descriptors once

This step computes features, not a classifier. No hypothesis test chooses whether
to retain a band. Both bands remain fixed inputs regardless of apparent direction.
Zero power remains zero; nonfinite, negative or entirely constant feature columns
stop execution rather than triggering a hidden fallback.

In [ ]:
features=pd.DataFrame([window_powers(x) for x in signals],columns=FEATURES,index=windows.index)
if not np.isfinite(features.to_numpy()).all() or (features<0).any().any():
    raise ValueError('Invalid band feature value.')
if features.nunique().lt(2).any():raise ValueError('A band is constant across all windows; inspect the descriptors.')
display(features.describe())
print('Model inputs:',FEATURES)
print('Units: squared CSV-header mV. Band RMS saved descriptively = square root of power.')


## 5. Execute frozen participant LOSO (real training; private Colab only)

The next cell fits15 SVMs and one final pipeline. Every test participant contributes
one evaluation, with both legs/all sets held out together. No test information
fits the imputer/scaler or selects parameters. SVC predictions retain sklearn's
default vote/tie behaviour. Scores are uncalibrated decision margins. Folds are
saved incrementally in a fresh timestamped directory; old outputs are untouched.
If a fit fails, the manifest stays failed with recoverable partial results.
There is no automatic run of older models or the expensive nested search.

In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=OUTPUT_ROOT/MODEL_KEY/stamp
OUT.mkdir(parents=True,exist_ok=False)
manifest=dict(status='running',model_key=MODEL_KEY,created_utc=stamp,config=CONFIG,
    feature_names=FEATURES,feature_units='CSV-header mV squared',
    input_hashes={name:sha(path) for name,path in paths.items()},
    source_processing_run=source['processing_run_id'],source_model_data_sha256=model_source['data_sha256'],
    versions={name:version(name) for name in PINS},python=platform.python_version(),
    device='CPU',platform=platform.platform(),validation='participant LOSO; no inner search',
    limitations=['Exploratory after earlier outer results were inspected.',
        'Fixed bands and band-power implementation are not independently validated fatigue labels.',
        'Exact spectral amplitude estimator in the quoted literature is not replicated.',
        'No MVC or independent physiological fatigue reference; offline filtered signals.',
        'Different feature sets do not isolate a causal physiological mechanism.'])
save_json(OUT/'run_manifest.json',manifest)
# Save a clean snapshot of the analysis definitions, even when training via notebook.
import inspect
definition_names=['integrate_band','window_powers','make_pipeline','evaluate']
snapshot=[]
for name in definition_names:
    try:snapshot.append(inspect.getsource(globals()[name]))
    except (OSError,TypeError):snapshot.append('# See saved executed notebook for '+name)
(OUT/'analysis_definition_snapshot.txt').write_text('\n\n'.join(snapshot),encoding='utf-8')
notebook_path=PROJECT_ROOT/'modelling/09_LFC_HFC_Fixed_RBF.ipynb'
if notebook_path.is_file():
    # Copy this notebook's on-disk source only; never export unrelated kernel history.
    nb_snapshot=json.loads(notebook_path.read_text(encoding='utf-8'))
    for cell in nb_snapshot.get('cells',[]):
        if cell.get('cell_type')=='code':cell['outputs']=[];cell['execution_count']=None
    save_json(OUT/'notebook_source_on_disk.ipynb',nb_snapshot)
    manifest['on_disk_notebook_sha256']=sha(notebook_path)
    manifest['source_snapshot_note']='On-disk copy; save the executed notebook separately, including any unsaved edits.'
dictionary=pd.DataFrame([dict(feature=FEATURES[0],family='Spectrum',units='header mV squared',band_hz='20–46.7'),
    dict(feature=FEATURES[1],family='Spectrum',units='header mV squared',band_hz='150–350')])
dictionary.to_csv(OUT/'feature_dictionary.csv',index=False)
export=windows.copy()
for name in FEATURES:export[name]=features[name]
export['LFC_band_RMS_descriptive_only']=np.sqrt(features[FEATURES[0]])
export['HFC_band_RMS_descriptive_only']=np.sqrt(features[FEATURES[1]])
export.to_csv(OUT/'band_features_and_metadata.csv',index=False)
y=windows.Label.to_numpy(dtype=int);groups=windows.subject_id.to_numpy()
folds=[];predictions=[];class_tables=[];leg_rows=[];splits=[]
started=time.perf_counter()
try:
    with threadpool_limits(limits=1):
        for fold,(train,test) in enumerate(LeaveOneGroupOut().split(features,y,groups),start=1):
            tick=time.perf_counter();subject=groups[test][0]
            assert subject not in set(groups[train])
            directory=OUT/f'fold_{fold:02d}_{subject}';directory.mkdir()
            split=dict(fold=fold,train_subjects=sorted(set(groups[train])),test_subjects=[subject],inner=[])
            save_json(directory/'split.json',split);splits.append(split)
            fitted=make_pipeline()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always');fitted.fit(features.iloc[train],y[train])
            warns=[dict(category=w.category.__name__,message=str(w.message)) for w in caught]
            save_json(directory/'fit_warnings.json',warns)
            if any(issubclass(w.category,ConvergenceWarning) for w in caught):raise RuntimeError('Unfinished SVM fit.')
            transformed=fitted.named_steps['impute'].transform(features.iloc[train])
            np.testing.assert_allclose(fitted.named_steps['scale'].mean_,transformed.mean(axis=0))
            pred=fitted.predict(features.iloc[test]);score=fitted.decision_function(features.iloc[test])
            result,pc=evaluate(y[test],pred,score)
            folds.append(dict(model=MODEL_KEY,fold=fold,test_subject=subject,n_train_subjects=len(set(groups[train])),
                n_train_windows=len(train),n_test_windows=len(test),n_features=2,selected_features='|'.join(FEATURES),
                C=1.,gamma='scale',seconds=time.perf_counter()-tick,**result))
            pc['fold']=fold;pc['subject_id']=subject;class_tables.append(pc)
            columns=['window_id','subject_id','recording_id','set_token','leg','sample_start','sample_end_exclusive','t_start_s','t_end_exclusive_s']
            oo=windows.iloc[test][columns].copy()
            oo['fold']=fold;oo['y_true']=y[test];oo['y_pred']=pred;oo['score']=score;oo['score_kind']='decision_function'
            predictions.append(oo)
            oo.to_csv(directory/'out_of_fold_predictions.csv',index=False)
            for leg,g in oo.groupby('leg'):
                values,_=evaluate(g.y_true,g.y_pred,g.score)
                leg_rows.append(dict(fold=fold,subject_id=subject,leg=leg,**values))
            joblib.dump(fitted,directory/'pipeline.joblib')
            save_json(directory/'nonlinear_model_details.json',dict(effective_gamma=float(fitted.named_steps['model']._gamma),
                support_vectors_per_class=fitted.named_steps['model'].n_support_.tolist()))
            pd.DataFrame(folds).to_csv(OUT/'partial_results.csv',index=False)
            print(f'{fold}/15 | {subject} | macro-F1={result["f1_macro"]:.3f}',flush=True)
        fold_metrics=pd.DataFrame(folds);oof=pd.concat(predictions,ignore_index=True)
        assert len(oof)==len(windows) and oof.window_id.is_unique and set(oof.window_id)==set(windows.window_id)
        fold_metrics.to_csv(OUT/'results.csv',index=False)
        fold_metrics.rename(columns={'test_subject':'subject_id'}).to_csv(OUT/'per_subject_metrics.csv',index=False)
        oof.to_csv(OUT/'out_of_fold_predictions.csv',index=False)
        pc=pd.concat(class_tables,ignore_index=True);pc.to_csv(OUT/'per_class_metrics.csv',index=False)
        pc.groupby('class_label')[['precision','recall','f1']].agg(['mean','std']).to_csv(OUT/'per_class_mean_sd.csv')
        legs=pd.DataFrame(leg_rows);legs.to_csv(OUT/'per_subject_leg_metrics.csv',index=False)
        legs.pivot(index='subject_id',columns='leg',values='f1_macro').to_csv(OUT/'paired_leg_f1.csv')
        summary=pd.DataFrame([dict(metric=m,mean=fold_metrics[m].mean(),sd=fold_metrics[m].std(ddof=1),
            n_defined=int(fold_metrics[m].notna().sum()),n_folds=len(fold_metrics)) for m in METRICS])
        summary.to_csv(OUT/'metrics_mean_sd.csv',index=False)
        pooled,pooled_pc=evaluate(oof.y_true,oof.y_pred,np.zeros(len(oof)))
        for key in ['roc_auc','pr_auc','average_precision']:pooled.pop(key)
        save_json(OUT/'pooled_window_metrics_secondary.json',pooled)
        pooled_pc.to_csv(OUT/'pooled_per_class_metrics.csv',index=False)
        save_json(OUT/'subject_splits.json',splits)
        if CONFIG['final_refit']:
            final_dir=OUT/'final_refit';final_dir.mkdir()
            final=make_pipeline()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always');final.fit(features,y)
            save_json(final_dir/'fit_warnings.json',[dict(category=w.category.__name__,message=str(w.message)) for w in caught])
            if any(issubclass(w.category,ConvergenceWarning) for w in caught):raise RuntimeError('Unfinished final fit.')
            joblib.dump(final,final_dir/'pipeline.joblib')
            save_json(final_dir/'model_card.json',dict(features=FEATURES,config=CONFIG,
                purpose='Full-data fit after evaluation; no additional independent test score.',
                upstream='Same offline filtered 500-sample windows; exact notebook Welch and band integration required.'))
        manifest.update(status='complete',n_outer_folds=len(folds),total_seconds=time.perf_counter()-started)
        save_json(OUT/'run_manifest.json',manifest)
except Exception as exc:
    manifest.update(status='failed',error=f'{type(exc).__name__}: {exc}')
    save_json(OUT/'run_manifest.json',manifest)
    raise
print('Completed run:',OUT)


## 6. Required results and descriptive error tables

Primary summaries are unweighted participant means ± sample SD over15 folds.
SD is not a confidence interval. AUCs use each participant's own model scores;
raw margins from different folds are not pooled for AUC. PR-AUC is trapezoidal;
AP is reported separately. Per-set/per-leg error summaries below are descriptive
and must not be used to select exclusions or tune this completed model.

In [ ]:
assert json.loads((OUT/'run_manifest.json').read_text())['status']=='complete'
shown=summary.copy()
shown['mean_plus_minus_sd']=shown.apply(lambda r:f'{r["mean"]:.3f} ± {r["sd"]:.3f}',axis=1)
display(shown[['metric','mean_plus_minus_sd','n_defined','n_folds']])
display(fold_metrics)
display(pc.groupby('class_label')[['precision','recall','f1']].agg(['mean','std']))
display(legs.pivot(index='subject_id',columns='leg',values='f1_macro'))
set_rows=[]
for (subject,rid,leg),g in oof.groupby(['subject_id','recording_id','leg']):
    values,_=evaluate(g.y_true,g.y_pred,g.score)
    set_rows.append(dict(subject_id=subject,recording_id=rid,leg=leg,**values))
set_metrics=pd.DataFrame(set_rows)
set_metrics.to_csv(OUT/'per_set_metrics_descriptive.csv',index=False)
display(set_metrics.sort_values('f1_macro').head(15))
print('Per-set rows are dependent within participant; participant LOSO remains the primary result.')

# Model reuse check on a trusted model created by this run; no fitting occurs here.
first=OUT/f'fold_01_{fold_metrics.iloc[0].test_subject}'
reloaded=joblib.load(first/'pipeline.joblib')
mask=windows.subject_id.eq(fold_metrics.iloc[0].test_subject)
np.testing.assert_array_equal(reloaded.predict(features.loc[mask]),oof.loc[oof.subject_id.eq(fold_metrics.iloc[0].test_subject),'y_pred'])
print('Trusted saved-pipeline predictions match the first held-out fold.')


In [ ]:
figures=OUT/'figures';figures.mkdir(exist_ok=True)
cm=confusion_matrix(oof.y_true,oof.y_pred,labels=[0,1])
norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
pd.DataFrame(cm,index=[0,1],columns=[0,1]).to_csv(OUT/'confusion_counts.csv')
pd.DataFrame(norm,index=[0,1],columns=[0,1]).to_csv(OUT/'confusion_row_normalized.csv')
fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
for ax,values,title,fmt in [(axes[0],cm,'OOF counts','d'),(axes[1],norm,'Row-normalized','.2f')]:
    ax.imshow(values,cmap='Blues')
    for i in [0,1]:
        for j in [0,1]:ax.text(j,i,format(values[i,j],fmt),ha='center',va='center')
    ax.set(xticks=[0,1],yticks=[0,1],xlabel='Predicted stage',ylabel='True stage',title=title)
fig.savefig(figures/'confusion_matrices.png',dpi=170);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4),layout='constrained')
ax.bar(fold_metrics.test_subject,fold_metrics.f1_macro)
ax.axhline(fold_metrics.f1_macro.mean(),color='black',linestyle='--')
ax.set(ylim=(0,1),ylabel='Macro-F1',xlabel='Held-out participant',title='Two-band fixed RBF')
ax.tick_params(axis='x',rotation=45)
fig.savefig(figures/'per_subject_f1.png',dpi=170);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for subject,g in oof.groupby('subject_id'):
    fpr,tpr,_=roc_curve(g.y_true,g.score);precision,recall,_=precision_recall_curve(g.y_true,g.score)
    axes[0].plot(fpr,tpr,alpha=.5,label=subject);axes[1].step(recall,precision,where='post',alpha=.5)
axes[0].plot([0,1],[0,1],'k--');axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='Per-participant ROC')
axes[1].set(xlabel='Recall late',ylabel='Precision late',title='Per-participant PR')
for ax in axes:ax.set(xlim=(0,1),ylim=(0,1.02))
axes[0].legend(fontsize=6,ncol=3)
fig.savefig(figures/'roc_pr_by_subject.png',dpi=170);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(11,4),layout='constrained')
ax.errorbar(np.arange(len(summary)),summary['mean'],yerr=summary.sd,fmt='o',capsize=4)
ax.set(xticks=np.arange(len(summary)),xticklabels=summary.metric,ylabel='Participant mean ± sample SD')
ax.tick_params(axis='x',rotation=50)
fig.savefig(figures/'metrics_mean_sd.png',dpi=170);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(11,3),layout='constrained');ax.axis('off')
ax.text(.5,.55,'15 LOSO folds: 14 training participants → fit imputer/scaler + fixed two-band SVM\n'
    '→ evaluate once on 1 held-out participant (both legs/all sets)\n'
    'No inner search; same fixed band definitions in every fold',ha='center',va='center',transform=ax.transAxes)
fig.savefig(figures/'validation_diagram.png',dpi=170);plt.show();plt.close(fig)


## 7. Descriptive stage changes and archive

The feature plot below summarizes paired set-stage medians, equal sets within
leg and equal legs within participant, requiring3 defined windows per stage and
both legs. It does not filter the features, test a fatigue diagnosis or select a
model. An expected trend in the literature is a hypothesis; report contrary or
inconsistent trends honestly. No automatic upload/download changes permissions.

In [ ]:
paired=windows[['subject_id','recording_id','leg','Label']].copy()
for name in FEATURES:paired[name]=features[name]
med=paired.groupby(['subject_id','leg','recording_id','Label'])[FEATURES].median()
counts=paired.groupby(['subject_id','leg','recording_id','Label'])[FEATURES].count()
delta=(med.xs(1,level='Label')-med.xs(0,level='Label')).where(
    (counts.xs(0,level='Label')>=3)&(counts.xs(1,level='Label')>=3))
by_leg=delta.groupby(level=['subject_id','leg']).mean()
by_subject=by_leg.groupby(level='subject_id').mean().where(by_leg.notna().groupby(level='subject_id').sum()==2)
by_subject.to_csv(OUT/'descriptive_participant_band_changes.csv')
display(by_subject)
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for ax,name in zip(axes,FEATURES):
    vals=by_subject[name].dropna()
    ax.bar(vals.index,vals);ax.axhline(0,color='black',lw=.8)
    ax.set(title=name,ylabel='Late − early power (header mV squared)');ax.tick_params(axis='x',rotation=60)
fig.savefig(figures/'descriptive_band_changes.png',dpi=170);plt.show();plt.close(fig)
archive=shutil.make_archive(str(OUT.parent/(OUT.name+'_results')),'zip',root_dir=OUT)
print('Private results archive:',archive)
print('Save this executed notebook separately. Existing runs/combined results were not overwritten.')


For report comparison, use `metrics_mean_sd.csv` and participant tables with
the earlier completed runs, retaining the same metric definitions. Do not choose
new bands or add ratios after seeing these outer results and describe them as
untouched validation. The final pipeline expects these exact two extracted
features in the saved order; apply the identical PSD/integration to new windows.
No physiological fatigue ground truth or improved accuracy is guaranteed.